<a href="https://colab.research.google.com/github/ttderessa/Temesgen-Deressa/blob/main/Agentic_AI_MCP_SandBox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# AUTHOR: Dr. Temesgen Deressa
# DATE: April 28, 2026
# PROJECT: Agentic AI Image Generation with Sandbox + MCP
# DESCRIPTION:
#   This script demonstrates a production-style AI agent that:
#   - Uses a sandbox (process isolation) for safe execution
#   - Simulates an MCP (Model Context Protocol) tool server
#   - Handles bulk requests with human approval
#   - Executes tasks in parallel for performance
#   - Provides robust error and timeout handling
# ============================================================


import asyncio
import multiprocessing
import time
from typing import List, Dict, Any


# ============================================================
# 1. SYSTEM CONFIGURATION
# ============================================================
# These configuration values define the operational limits of the system.
# They ensure the agent behaves safely and predictably by controlling:
# - when human approval is required (bulk threshold)
# - how long a task can run before being terminated (timeout)
# - how many parallel executions are allowed (scalability control)

BULK_THRESHOLD = 3
SANDBOX_TIMEOUT = 5.0
MAX_WORKERS = 4


# ============================================================
# 2. SANDBOX EXECUTION LAYER
# ============================================================
# The sandbox layer is the most important safety mechanism in this system.
# Instead of executing functions directly in the main process, each task
# runs in a completely separate process.
#
# This ensures:
# - Fault isolation (one failure does not crash the system)
# - Protection against infinite loops or long-running tasks
# - Safe handling of unexpected errors

def _sandbox_internal_wrapper(q, func, args):
    """
    Executes the function inside a separate process and returns results safely.
    """
    try:
        result = func(*args)
        q.put({"status": "success", "data": result})
    except Exception as e:
        q.put({"status": "error", "message": str(e)})


def sandbox_runner(func, args, timeout=SANDBOX_TIMEOUT):
    """
    Runs a function inside an isolated process with timeout protection.
    """
    q = multiprocessing.Queue()

    p = multiprocessing.Process(
        target=_sandbox_internal_wrapper,
        args=(q, func, args)
    )

    p.start()
    p.join(timeout)

    # If process runs too long, terminate it
    if p.is_alive():
        p.terminate()
        p.join()
        return {
            "status": "error",
            "message": f"Execution timed out after {timeout}s"
        }

    # If process fails to return output
    if q.empty():
        return {
            "status": "error",
            "message": "Process exited without returning data"
        }

    return q.get()


# ============================================================
# 3. MOCK MCP SERVER (SIMULATED TOOL)
# ============================================================
# This class simulates an external MCP tool server.
# In real production systems, this would be replaced by:
# - external APIs (image generation, LLM tools)
# - enterprise internal services
#
# It represents the "tool layer" that the agent interacts with.

class MockMCPServer:

    def generate_image(self, prompt: str) -> Dict[str, str]:
        # Simulate processing delay (real-world API latency)
        time.sleep(1.2)

        # Simulate controlled failure scenario for testing robustness
        if "fail" in prompt.lower():
            raise ValueError("Tool-level safety trigger: Content policy violation.")

        return {
            "prompt": prompt,
            "image_url": f"https://generated.ai/v1/{hash(prompt)}.png"
        }


# ============================================================
# 4. MCP TOOL EXECUTOR (SANDBOX + ASYNC BRIDGE)
# ============================================================
# This layer connects asynchronous Python execution with the blocking
# sandbox system. Since multiprocessing blocks the main thread,
# we use an async executor to keep the event loop responsive.
#
# It acts as a bridge between:
# - async agent logic
# - synchronous sandbox execution

class MCPToolExecutor:

    def __init__(self, server: MockMCPServer):
        self.server = server

    async def run_sandboxed(self, prompt: str):
        loop = asyncio.get_running_loop()

        return await loop.run_in_executor(
            None,
            sandbox_runner,
            self.server.generate_image,
            (prompt,)
        )


# ============================================================
# 5. IMAGE GENERATION AGENT
# ============================================================
# This is the core intelligence layer of the system.
# It is responsible for:
# - receiving user input (prompts)
# - enforcing safety rules (bulk approval)
# - dispatching sandboxed tool calls
# - collecting and reporting results
#
# It does NOT execute tools directly—everything goes through
# controlled sandbox execution.

class ImageGenerationAgent:

    def __init__(self, mcp_executor: MCPToolExecutor):
        self.mcp = mcp_executor

    async def handle_request(self, prompts: List[str]):
        count = len(prompts)
        print(f"🚀 Initializing request for {count} images...")

        # ----------------------------------------------------
        # SAFETY CONTROL: BULK APPROVAL
        # ----------------------------------------------------
        # EXPLANATION:
        # To prevent accidental large-scale execution, the system
        # requires human approval when the number of requests exceeds
        # a safe threshold.
        if count > BULK_THRESHOLD:
            if not await self.request_approval(count):
                print("🛑 Operation cancelled by user.")
                return

        # ----------------------------------------------------
        # PARALLEL SANDBOX EXECUTION
        # ----------------------------------------------------
        # Each prompt is executed independently in a sandbox.
        # This enables parallel processing while maintaining isolation
        # between tasks, improving both speed and reliability.
        print(f"⚙️ Dispatching {count} jobs to sandbox...")

        tasks = [self.mcp.run_sandboxed(p) for p in prompts]
        raw_results = await asyncio.gather(*tasks)

        # ----------------------------------------------------
        # RESULT PROCESSING
        # ----------------------------------------------------
        # After execution, results are normalized and displayed.
        # Both successful and failed executions are reported clearly
        # so the system remains transparent and debuggable.
        final_results = []

        print("\n--- Execution Report ---")

        for i, res in enumerate(raw_results):
            prompt = prompts[i]

            if res["status"] == "success":
                data = res["data"]
                print(f"✅ SUCCESS: [{prompt}] -> {data['image_url']}")
                final_results.append(data)
            else:
                print(f"❌ FAILED:  [{prompt}] -> {res['message']}")

        return final_results

    async def request_approval(self, count: int) -> bool:
        """
        Human-in-the-loop safety checkpoint.
        """
        print(f"\n⚠️ SAFETY GUARD: Bulk request detected ({count} items).")

        loop = asyncio.get_running_loop()

        answer = await loop.run_in_executor(
            None,
            lambda: input("👉 Proceed? (y/n): ").lower()
        )

        return answer == 'y'


# ============================================================
# 6. MAIN EXECUTION (COLAB / JUPYTER READY)
# ============================================================
# This section initializes all components and runs a sample workload.
# It includes both successful and failing prompts to demonstrate:
# - sandbox isolation
# - error handling
# - system resilience under mixed conditions

async def main():
    server = MockMCPServer()
    executor = MCPToolExecutor(server)
    agent = ImageGenerationAgent(executor)

    prompts = [
        "A cyberpunk cafe in the rain",
        "A landscape that will FAIL and trigger error",
        "Golden retriever astronaut",
        "Infinite loop simulation",
        "Abstract data visualizations"
    ]

    await agent.handle_request(prompts)


# ============================================================
# 7. RUN (COLAB / NOTEBOOK)
# ============================================================:
# Entry point for asynchronous execution environments such as
# Jupyter notebooks or Google Colab.

await main()

🚀 Initializing request for 5 images...

⚠️ SAFETY GUARD: Bulk request detected (5 items).
👉 Proceed? (y/n): y
⚙️ Dispatching 5 jobs to sandbox...

--- Execution Report ---
✅ SUCCESS: [A cyberpunk cafe in the rain] -> https://generated.ai/v1/-4505280474768062125.png
❌ FAILED:  [A landscape that will FAIL and trigger error] -> Tool-level safety trigger: Content policy violation.
✅ SUCCESS: [Golden retriever astronaut] -> https://generated.ai/v1/-1401190453259258164.png
✅ SUCCESS: [Infinite loop simulation] -> https://generated.ai/v1/8970881257958609265.png
✅ SUCCESS: [Abstract data visualizations] -> https://generated.ai/v1/-6204001673508885656.png
